# 🧬 ADSP × Biofilter 4 — from 1.3 million variants to pairs worth testing

Testing every pair of 1.3 million variants for interaction would mean almost
**10¹² models**. This notebook narrows that down to the pairs that biology
gives a reason to test. It takes three steps, each one a question for
Biofilter:

| step | question | Biofilter report |
| --- | --- | --- |
| **1** | Does the variant touch a protein-coding gene? | `annotate_variant`, `annotate_gene` |
| **2** | Does it actually do something to that gene? | `expand_variant_regulatory` |
| **3** | Which pairs of those genes share a pathway? | `pair_genes` |

**What you need:** a Biofilter bundle (a folder of Parquet files) and the
variant list. There is no database to install and nothing to import. Every
step below runs on a laptop.

## 0 · Setup

In [1]:
from pathlib import Path
import pandas as pd
from biofilter import Biofilter
from biofilter.utils.logger import Logger

Logger(log_level="WARNING")   # quiet: show warnings only

BUNDLE   = "../../../biofilter_data/bundles/20260914"   # a bundle is just a folder
VARIANTS = "ADSP_R5.final_LD_pruned_variants_biofilter_input.txt"
PQTL     = "../adsp/data/brain_pQTL_hmt_variant_gene_lookup.tsv.gz"  # FunGen xQTL, not in Biofilter
RESULTS  = Path("results"); RESULTS.mkdir(exist_ok=True)

bf = Biofilter(bundle=BUNDLE)

## 1 · The input

One variant per line, written as `chromosome:position:ref:alt`.

In [2]:
variants = Path(VARIANTS).read_text().split()
print(f"{len(variants):,} variants   e.g. {variants[:3]}")

1,312,862 variants   e.g. ['1:633963:C:T', '1:702358:G:A', '1:722408:G:C']

## 2 · Step 1 — does the variant touch a protein-coding gene?

Two reports, each answering half of the question:

- `annotate_variant` returns **which genes** VEP links each variant to, for
  the gene the variant lies in and for genes up to 5 kb upstream or
  downstream.
- `annotate_gene` returns **what kind of gene** each of those is, according to
  HGNC.

In [3]:
vep = bf.report.run("annotate_variant", input_data=variants).to_pandas()
vep = vep[vep.status != "not_found"]
vep["gene"] = vep.gene_symbol.fillna(vep.gene_id)

print(f"{vep.input_value.nunique():,} of {len(variants):,} variants found in the bundle")

1,312,677 of 1,312,862 variants found in the bundle

In [4]:
genes = bf.report.run(
    "annotate_gene",
    input_data=vep.gene.dropna().unique().tolist(),
    include_variant_summary=False,
).to_pandas()

coding_genes = genes[
    genes.gene_locus_group.eq("protein-coding gene")
    & genes.gene_locus_type.notna()
    & genes.gene_locus_type.ne("unknown")
][["input_value", "entity_id", "ensembl_id", "gene_symbol"]]

# Keep each variant's most severe consequence for each coding gene it touches
step1 = (
    vep.merge(coding_genes, left_on="gene", right_on="input_value", suffixes=("", "_gene"))
       .sort_values("severity_rank")
       .drop_duplicates(["input_value", "entity_id"])
       .rename(columns={"input_value": "variant"})
)
print(f"Step 1 → {step1.variant.nunique():,} variants on a protein-coding gene")

Step 1 → 655,036 variants on a protein-coding gene

## 3 · Step 2 — does it actually do something to that gene?

"Functional" means different things for coding and non-coding variants, so
each variant goes into **exactly one** of two queues:

- **Queue A, coding:** the variant must **change the protein**. This means
  VEP severity rank ≤ 14 (missense, stop gained or lost, start lost, splice
  acceptor or donor, frameshift, in-frame indels).
- **Queue B, non-coding:** the variant must regulate a gene at **both the RNA
  and the protein level**: a brain eQTL (GTEx, from the bundle) and a brain
  pQTL (FunGen) for **the same gene**. The variant is then paired on that
  target gene, which is often not the gene it lies in.

In [5]:
is_coding = step1.consequence_category.eq("coding")
coding_variants = set(step1.loc[is_coding, "variant"])

queue_a = step1[is_coding & step1.severity_rank.le(14)]
queue_a = pd.DataFrame({"variant": queue_a.variant, "gene": queue_a.ensembl_id,
                        "symbol": queue_a.gene_symbol, "queue": "A"})

print(f"Queue A → {queue_a.variant.nunique():,} protein-altering variants")

Queue A → 5,248 protein-altering variants

In [6]:
non_coding = sorted(set(step1.variant) - coding_variants)

eqtl = bf.report.run("expand_variant_regulatory", input_data=non_coding).to_pandas()
eqtl = eqtl[eqtl.status.eq("ok") & eqtl.regulated_gene_id.notna()]

pqtl = pd.read_csv(PQTL, sep="\t")
pqtl["variant"] = pqtl.variant_id.str.removeprefix("chr")

queue_b = (
    eqtl.merge(pqtl, left_on=["input_value", "regulated_gene_id"],
                     right_on=["variant", "target_ensembl_id"])
        .drop_duplicates(["variant", "regulated_gene_id"])
)
queue_b = pd.DataFrame({"variant": queue_b.variant, "gene": queue_b.regulated_gene_id,
                        "symbol": queue_b.regulated_gene_symbol, "queue": "B"})

print(f"Queue B → {queue_b.variant.nunique():,} regulatory variants (eQTL + pQTL on the same gene)")

Queue B → 8,917 regulatory variants (eQTL + pQTL on the same gene)

In [7]:
step2 = pd.concat([queue_a, queue_b]).dropna(subset=["gene"]).drop_duplicates(["variant", "gene"])
step2.to_csv(RESULTS / "selected_variants.csv", index=False)

print(f"Step 2 → {step2.variant.nunique():,} variants, on {step2.gene.nunique():,} genes")
step2.head()

Step 2 → 14,165 variants, on 4,569 genes

,variant,gene,symbol,queue
3270374,16:87730661:A:T,ENSG00000104731,KLHDC4,A
4282064,19:41586462:A:T,ENSG00000007129,CEACAM21,A
4541376,1:11682017:C:T,ENSG00000116670,MAD2L2,A
5405407,20:35626801:T:C,ENSG00000214078,CPNE1,A
3246668,16:85185117:A:G,ENSG00000131149,GSE1,A


## 4 · Step 3 — which pairs share a pathway?

`pair_genes` takes the genes, along with the variants we chose for each one,
and returns every pair of variants whose genes appear together in a curated
pathway. Two settings:

- `min_group_sources=2`: Reactome **and** KEGG must both support the pair.
- `max_group_size=500`: pathways naming more than 500 genes are ignored,
  because they link almost everything and say little about any one pair.

In [8]:
variants_by_gene = step2.groupby("gene").variant.apply(list).to_dict()

pairs = bf.report.run(
    "pair_genes",
    input_data=sorted(variants_by_gene),
    mapping=variants_by_gene,
    gene_identifier="ensembl",
    min_group_sources=2,
    max_group_size=500,
)
models = pairs.to_pandas()
print(f"Step 3 → {len(models):,} variant pairs to test")

[WARNING] ⚠️  pair_genes: 13 name(s) in the mapping answer to more than one gene, so their items attach to each. Supply an entity id to say which you meant.

[WARNING] ⚠️  pair_genes: 1 gene(s) in the mapping do not resolve to a gene in this bundle, so their items cannot be paired.

Step 3 → 312,991 variant pairs to test

In [9]:
models[["item_1", "item_2", "gene_1_symbol", "gene_2_symbol",
        "group_support_count", "group_support_sources"]].head(10)

,item_1,item_2,gene_1_symbol,gene_2_symbol,group_support_count,group_support_sources
0,14:23034812:G:A,17:38751870:C:T,PSMB3,PSMB5,165,"[kegg_relationships, reactome_relationships]"
1,14:23034812:G:A,17:38752259:C:G,PSMB3,PSMB5,165,"[kegg_relationships, reactome_relationships]"
2,14:23034812:G:A,17:38768086:G:A,PSMB3,PSMB5,165,"[kegg_relationships, reactome_relationships]"
3,14:23034812:G:A,9:124414882:A:G,PSMB5,PSMB7,165,"[kegg_relationships, reactome_relationships]"
4,17:38751870:C:T,9:124414882:A:G,PSMB3,PSMB7,165,"[kegg_relationships, reactome_relationships]"
5,17:38752259:C:G,9:124414882:A:G,PSMB3,PSMB7,165,"[kegg_relationships, reactome_relationships]"
6,17:38768086:G:A,9:124414882:A:G,PSMB3,PSMB7,165,"[kegg_relationships, reactome_relationships]"
7,3:184302754:G:T,9:124414882:A:G,PSMB7,PSMD2,165,"[kegg_relationships, reactome_relationships]"
8,14:23034812:G:A,3:184302754:G:T,PSMB5,PSMD2,164,"[kegg_relationships, reactome_relationships]"
9,17:38751870:C:T,3:184302754:G:T,PSMB3,PSMD2,164,"[kegg_relationships, reactome_relationships]"


## 5 · Save

`write()` saves the table and, next to it, a `.provenance.json` that records
which bundle, which report and which parameters produced it. Anyone can then
reproduce the file exactly.

In [10]:
pairs.write(RESULTS / "variant_pairs_2src.csv")
print(pairs.provenance["bundle_id"])

9b8419b48be5004e

## 6 · Summary

In [11]:
pd.DataFrame({
    "step": ["input", "1 · protein-coding gene", "2 · functional (A + B)", "3 · variant pairs"],
    "count": [len(variants), step1.variant.nunique(), step2.variant.nunique(), len(models)],
}).style.format({"count": "{:,}"}).hide(axis="index")

step,count
input,"1,312,862"
1 · protein-coding gene,"655,036"
2 · functional (A + B),"14,165"
3 · variant pairs,"312,991"


**That's the whole pipeline: four report calls on one folder of files.**

Want to try other choices? Change one argument and re-run the cell:

- `min_group_sources=1` accepts a pathway from a single source (a much larger set).
- `max_group_size=0` removes the pathway size limit.
- `group_types=["Pathways", "Diseases"]` also pairs genes that share a disease.
- `bf.report.explain("pair_genes")` describes every option.